# echopype Pipeline — Raw Files to Training-Ready Dataset

This notebook converts Simrad EK80 `.raw` files into the same standardized zarr
format as the CRIMAC pipeline (notebook 13), but using echopype for calibration
and echopype's built-in `detect_seafloor` for bottom detection.

Each processing step is shown explicitly so the calibration and detection logic
is fully transparent and verifiable.

**Note:** `detect_seafloor` requires a development install of echopype
(`pip install -e ".[dev]"` from the echopype source repo).

The workflow:
1. Open a `.raw` file with echopype
2. Compute calibrated S_v (TVG, absorption, gain, beam corrections)
3. Add depth and extract heave from MRU0 datagrams
4. Run seafloor detection (basic threshold or Blackwell method)
5. Create bottom mask and save to standardized zarr
6. Batch process all files
7. Verification dashboard

## 1. Configuration

In [ ]:
import sys
sys.path.insert(0, '../..')

from pathlib import Path

import echopype as ep
from echopype.mask import detect_seafloor
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from tqdm.auto import tqdm

from raw_sensor_integration import raw_parser, bottom_detection
from raw_sensor_integration.bottom_detection import (
    bottom_to_sample_index, create_bottom_mask,
)
from raw_sensor_integration.data_loader import save_dataset_zarr, load_dataset_zarr

plt.rcParams['figure.figsize'] = (16, 5)
plt.rcParams['image.cmap'] = 'viridis'

In [ ]:
# ── Paths ──
RAW_DIR = Path('/Users/axelhagen/Master/Datasets/raw_files/drive-download-20260317T113707Z-1-001/')
OUTPUT_DIR = Path('/Users/axelhagen/Master/Datasets/echopype_dataset/')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── echopype settings ──
WAVEFORM_MODE = 'BB'       # 'BB' for broadband, 'CW' for continuous wave
ENCODE_MODE = 'complex'    # standard EK80 encoding
DEPTH_OFFSET = 0.0         # transducer depth below waterline (m) — adjust per vessel

# ── Seafloor detection (echopype.mask.detect_seafloor) ──
# Method: 'basic' (threshold-based) or 'blackwell' (uses split-beam angles)
DETECTION_METHOD = 'basic'
DETECTION_PARAMS = dict(
    var_name='Sv',
    threshold=(-40, -20),
    offset_m=0.3,
    bin_skip_from_surface=100,
)

# To use the Blackwell method instead, uncomment below and comment out the above:
# DETECTION_METHOD = 'blackwell'
# DETECTION_PARAMS = dict(
#     var_name='Sv',
#     threshold=[-40, 702, 282],
#     offset=0.3,
#     r0=10, r1=1000,
#     wtheta=28, wphi=52,
# )

# Skip files that already have output zarrs
SKIP_EXISTING = True

## 2. File discovery

In [ ]:
raw_files = sorted(RAW_DIR.glob('*.raw'))
print(f'{len(raw_files)} .raw files in {RAW_DIR.name}')
for f in raw_files:
    size_mb = f.stat().st_size / 1e6
    print(f'  {f.name}  ({size_mb:.0f} MB)')

## 3. Step-by-step processing of one file

Each step is in its own cell so you can see exactly what happens during
calibration and detection.

### 3a. Open raw file with echopype

`echopype.open_raw()` parses all datagrams (XML config, RAW3 samples, NME0 GPS,
MRU0 motion) into an EchoData object without performing any calibration.

In [ ]:
demo_file = raw_files[0]
print(f'Processing: {demo_file.name}')

ed = ep.open_raw(str(demo_file), sonar_model='EK80')
print(f'\nEchoData groups:')
print(f'  Top-level:    {list(ed.top.data_vars)}')
print(f'  Environment:  {list(ed.environment.data_vars)}')
print(f'  Platform:     {list(ed.platform.data_vars)}')
print(f'  Beam groups:  {len(ed.beam.channel)} channels')
for ch in ed.beam.channel.values:
    print(f'    {ch}')

### 3b. Compute calibrated S_v

`echopype.calibrate.compute_Sv()` applies time-varied gain (TVG), absorption
correction, transducer gain, and beam-angle corrections to produce calibrated
volume backscattering strength (S_v) in dB re 1 m^-1.

In [ ]:
ds_Sv = ep.calibrate.compute_Sv(ed, waveform_mode=WAVEFORM_MODE, encode_mode=ENCODE_MODE)

# Add depth coordinate (required by detect_seafloor)
ds_Sv = ep.consolidate.add_depth(ds_Sv, ed, depth_offset=DEPTH_OFFSET)

print(f'Calibrated Sv dataset:')
print(f'  Dimensions: {dict(ds_Sv.sizes)}')
print(f'  Channels:   {ds_Sv.channel.values}')
print(f'  Frequencies: {ds_Sv.frequency_nominal.values} Hz')
print(f'  Sv range:   [{float(ds_Sv.Sv.min()):.1f}, {float(ds_Sv.Sv.max()):.1f}] dB')
print(f'  Has depth:  {"depth" in ds_Sv}')

### 3c. Extract S_v and range arrays per channel

echopype stores S_v with dimensions `(channel, ping_time, range_sample)` and
range in the `echo_range` variable. We transpose to our convention
`(n_range, n_pings)` and trim trailing NaN range samples.

In [ ]:
n_channels = len(ds_Sv.channel)
frequencies = ds_Sv.frequency_nominal.values.astype(np.float64)
channel_ids = [str(c) for c in ds_Sv.channel.values]

# Build per-channel Sv arrays and find common range
sv_per_channel = []
range_per_channel = []

for ci in range(n_channels):
    sv_raw = ds_Sv.Sv.isel(channel=ci).values  # (ping_time, range_sample) — already in dB
    range_2d = ds_Sv.echo_range.isel(channel=ci).values  # (ping_time, range_sample)
    range_1d = np.nanmean(range_2d, axis=0)  # average across pings

    # Trim trailing NaN range values
    valid_range = np.isfinite(range_1d)
    if valid_range.any():
        last_valid = np.where(valid_range)[0][-1] + 1
        range_1d = range_1d[:last_valid]
        sv_raw = sv_raw[:, :last_valid]

    sv_per_channel.append(sv_raw.T.astype(np.float32))  # (n_range, n_pings)
    range_per_channel.append(range_1d)

    print(f'Channel {ci} ({frequencies[ci]/1000:.0f} kHz): '
          f'Sv shape {sv_raw.T.shape}, range 0–{range_1d[-1]:.1f} m')

# Align to shortest range (channels may have different range extents)
min_range_len = min(len(r) for r in range_per_channel)
range_array = range_per_channel[0][:min_range_len]
sv_3d = np.stack([sv[:min_range_len, :] for sv in sv_per_channel])  # (n_freq, n_range, n_pings)

# Clip extreme noise floor values (echopype can produce values below -200 dB)
sv_3d = np.clip(sv_3d, -120.0, None)

n_pings = sv_3d.shape[2]
timestamps = ds_Sv.ping_time.values

print(f'\nAligned shape: {sv_3d.shape}  (freq, range, pings)')
print(f'Sv after clipping: [{np.nanmin(sv_3d):.1f}, {np.nanmax(sv_3d):.1f}] dB')

### 3d. Extract heave from MRU0 datagrams

echopype does not extract heave directly. We parse the MRU0 motion datagrams
from the raw binary file and interpolate heave values to the ping timestamps.

In [ ]:
raw_data = raw_parser.read_raw_file(str(demo_file))
datagram_types = raw_parser.find_datagram_types(raw_data)

ping_times_dt = pd.to_datetime(timestamps).to_pydatetime().tolist()

if 'MRU0' in datagram_types and len(datagram_types['MRU0']) > 0:
    mru_records = bottom_detection.parse_mru0_datagrams(raw_data, datagram_types['MRU0'])
    mru_times = [r[0] for r in mru_records]
    heave_values = [r[1] for r in mru_records]
    heave = bottom_detection.interpolate_heave_to_pings(
        mru_times, heave_values, ping_times_dt)
    print(f'Heave: {len(mru_records)} MRU0 records → interpolated to {n_pings} pings')
    print(f'  range: [{heave.min():.3f}, {heave.max():.3f}] m')
else:
    heave = np.zeros(n_pings, dtype=np.float64)
    print('No MRU0 datagrams found — heave set to zero')

del raw_data  # free memory

### 3e. Visualize the calibrated echogram

In [ ]:
n_freq = sv_3d.shape[0]
fig, axes = plt.subplots(n_freq, 1, figsize=(16, 4 * n_freq), sharex=True, squeeze=False)
ping_idx = np.arange(n_pings)

for fi in range(n_freq):
    ax = axes[fi, 0]
    ax.imshow(sv_3d[fi], aspect='auto', origin='upper', vmin=-80, vmax=-34,
              extent=[ping_idx[0], ping_idx[-1], range_array[-1], range_array[0]])
    ax.set_ylabel('Range (m)')
    ax.set_title(f'{frequencies[fi]/1000:.0f} kHz — {demo_file.name}')

axes[-1, 0].set_xlabel('Ping index')
plt.tight_layout()
plt.show()

### 3f. Run seafloor detection with echopype

`echopype.mask.detect_seafloor()` operates on the calibrated Sv dataset directly.
The `basic` method applies a threshold search from the bottom of the echogram upward.
The `blackwell` method additionally uses split-beam angles for more robust detection
when angles are available.

In [ ]:
# Select detection channel (first frequency)
det_channel = str(ds_Sv.channel.values[0])
params = {**DETECTION_PARAMS, 'channel': det_channel}

# For blackwell: merge split-beam angles into ds_Sv
if DETECTION_METHOD == 'blackwell':
    for beam_grp in ['Sonar/Beam_group1', 'Sonar/Beam_group2']:
        if beam_grp in ed.group_paths:
            angle_along = ed[beam_grp]['angle_alongship']
            angle_athwart = ed[beam_grp]['angle_athwartship']
            ds_Sv = ds_Sv.assign(
                angle_alongship=angle_along,
                angle_athwartship=angle_athwart,
            )
            break

bottom_depth = detect_seafloor(ds_Sv, method=DETECTION_METHOD, params=params)

# Convert absolute depth → range from transducer
bottom_range = bottom_depth.values - DEPTH_OFFSET

valid = np.isfinite(bottom_range)
print(f'Detection method: {DETECTION_METHOD}')
print(f'Channel: {det_channel}')
print(f'Valid pings: {valid.sum()}/{n_pings} ({100*valid.mean():.1f}%)')
print(f'Bottom range: [{np.nanmin(bottom_range):.1f}, {np.nanmax(bottom_range):.1f}] m')

### 3g. Visualize detected seafloor

In [ ]:
sv_primary = sv_3d[0]  # first frequency, (n_range, n_pings)
extent = [ping_idx[0], ping_idx[-1], range_array[-1], range_array[0]]

fig, ax = plt.subplots(figsize=(16, 5))
ax.imshow(sv_primary, aspect='auto', origin='upper', vmin=-80, vmax=-34, extent=extent)

valid = np.isfinite(bottom_range)
ax.plot(ping_idx[valid], bottom_range[valid], color='red', lw=1.2, alpha=0.9,
        label=f'{DETECTION_METHOD} detector')

ax.set_ylabel('Range (m)')
ax.set_xlabel('Ping index')
ax.set_title(f'Seafloor detection ({DETECTION_METHOD}) — {demo_file.name}')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

### 3h. Create mask and save to standardized zarr

In [ ]:
bottom_indices = bottom_to_sample_index(bottom_range, range_array)
mask = create_bottom_mask(len(range_array), n_pings, bottom_indices, mode='below')

# Show mask overlaid on echogram
fig, ax = plt.subplots(figsize=(16, 5))
ax.imshow(sv_primary, aspect='auto', origin='upper', vmin=-80, vmax=-34, extent=extent)
ax.imshow(mask, aspect='auto', origin='upper', extent=extent,
          cmap='Reds', alpha=0.2)
valid = np.isfinite(bottom_range)
ax.plot(np.where(valid)[0], bottom_range[valid], 'r-', lw=1)
ax.set_ylabel('Range (m)')
ax.set_xlabel('Ping index')
ax.set_title(f'Bottom mask (below) — {demo_file.name}')
plt.tight_layout()
plt.show()

print(f'Mask shape: {mask.shape}, sum: {mask.sum():,} pixels ({100*mask.mean():.1f}% of image)')

In [ ]:
# Save to standardized format
demo_output = OUTPUT_DIR / f'{demo_file.stem}.zarr'
save_dataset_zarr(
    sv_db=sv_3d,
    frequencies=frequencies,
    range_array=range_array,
    timestamps=timestamps,
    bottom_range=bottom_range,
    bottom_mask=mask,
    heave=heave.astype(np.float32),
    output_path=demo_output,
    pipeline='echopype',
    detection_algorithm=f'echopype {DETECTION_METHOD}',
    detection_params=DETECTION_PARAMS,
    source_files=[demo_file.name],
    channel_ids=channel_ids,
)
print(f'Saved: {demo_output}')

## 4. Batch processing

Process all `.raw` files through the same pipeline shown step-by-step above.
Each file is opened with echopype, calibrated, run through bottom detection,
and saved to a standardized zarr.

In [ ]:
summaries = []
failed = []

for raw_path in tqdm(raw_files, desc='Processing .raw files'):
    output_path = OUTPUT_DIR / f'{raw_path.stem}.zarr'

    if SKIP_EXISTING and output_path.exists():
        print(f'  skip (exists): {output_path.name}')
        continue

    try:
        # 1. Open, calibrate, add depth
        ed = ep.open_raw(str(raw_path), sonar_model='EK80')
        ds_Sv = ep.calibrate.compute_Sv(ed, waveform_mode=WAVEFORM_MODE, encode_mode=ENCODE_MODE)
        ds_Sv = ep.consolidate.add_depth(ds_Sv, ed, depth_offset=DEPTH_OFFSET)

        # 2. Extract Sv per channel, align to common range
        n_ch = len(ds_Sv.channel)
        freqs = ds_Sv.frequency_nominal.values.astype(np.float64)
        ch_ids = [str(c) for c in ds_Sv.channel.values]

        sv_list = []
        rng_list = []
        for ci in range(n_ch):
            sv_raw = ds_Sv.Sv.isel(channel=ci).values
            range_2d = ds_Sv.echo_range.isel(channel=ci).values
            range_1d = np.nanmean(range_2d, axis=0)
            valid = np.isfinite(range_1d)
            if valid.any():
                last = np.where(valid)[0][-1] + 1
                range_1d = range_1d[:last]
                sv_raw = sv_raw[:, :last]
            sv_list.append(sv_raw.T.astype(np.float32))
            rng_list.append(range_1d)

        min_len = min(len(r) for r in rng_list)
        rng = rng_list[0][:min_len]
        sv = np.stack([s[:min_len, :] for s in sv_list])
        sv = np.clip(sv, -120.0, None)

        n_p = sv.shape[2]
        ts = ds_Sv.ping_time.values

        # 3. Heave from MRU0
        raw_bytes = raw_parser.read_raw_file(str(raw_path))
        dg_types = raw_parser.find_datagram_types(raw_bytes)
        pt_dt = pd.to_datetime(ts).to_pydatetime().tolist()

        if 'MRU0' in dg_types and len(dg_types['MRU0']) > 0:
            mru = bottom_detection.parse_mru0_datagrams(raw_bytes, dg_types['MRU0'])
            hv = bottom_detection.interpolate_heave_to_pings(
                [r[0] for r in mru], [r[1] for r in mru], pt_dt)
        else:
            hv = np.zeros(n_p, dtype=np.float64)
        del raw_bytes

        # 4. Seafloor detection with echopype
        det_ch = str(ds_Sv.channel.values[0])
        det_params = {**DETECTION_PARAMS, 'channel': det_ch}

        if DETECTION_METHOD == 'blackwell':
            for beam_grp in ['Sonar/Beam_group1', 'Sonar/Beam_group2']:
                if beam_grp in ed.group_paths:
                    ds_Sv = ds_Sv.assign(
                        angle_alongship=ed[beam_grp]['angle_alongship'],
                        angle_athwartship=ed[beam_grp]['angle_athwartship'],
                    )
                    break

        bottom_depth = detect_seafloor(ds_Sv, method=DETECTION_METHOD, params=det_params)
        br = bottom_depth.values - DEPTH_OFFSET

        # 5. Mask and save
        bi = bottom_to_sample_index(br, rng)
        msk = create_bottom_mask(len(rng), n_p, bi, mode='below')

        save_dataset_zarr(
            sv_db=sv, frequencies=freqs, range_array=rng, timestamps=ts,
            bottom_range=br, bottom_mask=msk, heave=hv.astype(np.float32),
            output_path=output_path, pipeline='echopype',
            detection_algorithm=f'echopype {DETECTION_METHOD}',
            detection_params=DETECTION_PARAMS,
            source_files=[raw_path.name], channel_ids=ch_ids,
        )

        valid_pct = 100 * np.mean(np.isfinite(br))
        summaries.append({
            'file': raw_path.name,
            'n_pings': n_p,
            'n_freq': n_ch,
            'n_range': len(rng),
            'valid_bottom': f'{valid_pct:.1f}%',
            'depth_median': f'{np.nanmedian(br):.1f} m',
            'output': output_path.name,
        })

        # Free memory
        del ed, ds_Sv, sv, rng, hv, br, msk

    except Exception as e:
        print(f'  FAILED: {raw_path.name} — {e}')
        failed.append({'file': raw_path.name, 'error': str(e)})

print(f'\nDone: {len(summaries)} succeeded, {len(failed)} failed')
pd.DataFrame(summaries)

In [ ]:
if failed:
    print('Failed files:')
    pd.DataFrame(failed)

## 5. Verification dashboard

Load the saved zarrs back and show echograms with detected bottom lines for
a quick visual check.

In [ ]:
zarr_files = sorted(OUTPUT_DIR.glob('*.zarr'))
n_files = len(zarr_files)
n_cols = min(2, n_files)
n_rows = (n_files + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows), squeeze=False)

for idx, zp in enumerate(zarr_files):
    ax = axes[idx // n_cols, idx % n_cols]
    data = load_dataset_zarr(zp)

    sv = data['sv_db'][0]  # first frequency
    rng = data['range_array']
    br = data['bottom_range']

    ax.imshow(sv, aspect='auto', origin='upper', vmin=-80, vmax=-34,
              extent=[0, sv.shape[1], rng[-1], rng[0]])

    valid = np.isfinite(br)
    ax.plot(np.where(valid)[0], br[valid], 'r-', lw=0.8, alpha=0.9)

    valid_pct = 100 * valid.mean()
    freq_khz = data['frequencies'][0] / 1000
    ax.set_title(f'{zp.name}  ({freq_khz:.0f} kHz, {valid_pct:.0f}% valid)', fontsize=9)
    ax.set_ylabel('Range (m)')

for idx in range(n_files, n_rows * n_cols):
    axes[idx // n_cols, idx % n_cols].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Round-trip verification on the first zarr
test_data = load_dataset_zarr(zarr_files[0])
print(f'Loaded: {zarr_files[0].name}')
print(f'  sv_db shape:       {test_data["sv_db"].shape}  (freq, range, pings)')
print(f'  frequencies:       {test_data["frequencies"]} Hz')
print(f'  range:             0 – {test_data["range_array"][-1]:.1f} m')
print(f'  bottom_range:      {np.nanmin(test_data["bottom_range"]):.1f} – {np.nanmax(test_data["bottom_range"]):.1f} m')
print(f'  bottom_mask sum:   {test_data["bottom_mask"].sum():,} pixels')
print(f'  pipeline:          {test_data["attrs"]["pipeline"]}')
print(f'  detector:          {test_data["attrs"]["detection_algorithm"]}')

## 6. Comparison with CRIMAC pipeline

If CRIMAC-processed zarrs exist for the same files (from notebook 13), load
both versions and compare calibrated S_v values and bottom detection results.

In [ ]:
CRIMAC_DIR = Path('/Users/axelhagen/Master/Datasets/crimac_dataset/')

if CRIMAC_DIR.exists():
    matched = 0
    for zp in zarr_files:
        crimac_path = CRIMAC_DIR / zp.name
        if not crimac_path.exists():
            continue
        matched += 1

        ep_data = load_dataset_zarr(zp)
        crimac_data = load_dataset_zarr(crimac_path)

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        # echopype echogram + bottom
        sv_e = ep_data['sv_db'][0]
        rng_e = ep_data['range_array']
        br_e = ep_data['bottom_range']
        axes[0].imshow(sv_e, aspect='auto', origin='upper', vmin=-80, vmax=-34,
                       extent=[0, sv_e.shape[1], rng_e[-1], rng_e[0]])
        valid_e = np.isfinite(br_e)
        axes[0].plot(np.where(valid_e)[0], br_e[valid_e], 'r-', lw=1)
        axes[0].set_title(f'echopype — {zp.name}')
        axes[0].set_ylabel('Range (m)')

        # CRIMAC echogram + bottom
        sv_c = crimac_data['sv_db'][0]
        rng_c = crimac_data['range_array']
        br_c = crimac_data['bottom_range']
        axes[1].imshow(sv_c, aspect='auto', origin='upper', vmin=-80, vmax=-34,
                       extent=[0, sv_c.shape[1], rng_c[-1], rng_c[0]])
        valid_c = np.isfinite(br_c)
        axes[1].plot(np.where(valid_c)[0], br_c[valid_c], 'r-', lw=1)
        axes[1].set_title(f'CRIMAC — {zp.name}')

        # Bottom depth difference histogram
        n_common = min(len(br_e), len(br_c))
        diff = ep_data['bottom_range'][:n_common] - crimac_data['bottom_range'][:n_common]
        both_valid = np.isfinite(diff)
        if both_valid.any():
            axes[2].hist(diff[both_valid], bins=50, edgecolor='black', alpha=0.7)
            axes[2].axvline(0, color='red', ls='--')
            axes[2].set_xlabel('echopype - CRIMAC bottom depth (m)')
            med = np.nanmedian(diff)
            mad = np.nanmedian(np.abs(diff[both_valid]))
            axes[2].set_title(f'Median diff: {med:.2f} m, MAD: {mad:.2f} m')
        else:
            axes[2].text(0.5, 0.5, 'No overlapping valid pings',
                         ha='center', va='center', transform=axes[2].transAxes)

        plt.tight_layout()
        plt.show()

    if matched == 0:
        print('No matching filenames found between echopype and CRIMAC outputs.')
        print(f'echopype files: {[z.name for z in zarr_files]}')
        crimac_files = sorted(CRIMAC_DIR.glob('*.zarr'))
        print(f'CRIMAC files:  {[z.name for z in crimac_files]}')
else:
    print(f'CRIMAC output directory not found: {CRIMAC_DIR}')
    print('Run notebook 13 first, then re-run this cell.')